# Milestone 0: Dataset Selection & Quality Audit

**Dataset:** NOAA Global Historical Climatology Network Daily (GHCN-Daily)

Daily weather measurements (temperature, precipitation, snow and more) from over 100,000 stations around the world, with records ranging from 1763 to 2026.

---

### Documentation

**Data source (one file per year):**
- https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year

**Data dictionary:**

- **Raw yearly files:**
   - **STATION**: 11-character station ID (first 2 letters are the country code, e.g. EI = Ireland)
   - **DATE**: date of the reading, as YYYYMMDD
   - **ELEMENT**: type of measurement, e.g. TMAX (max temperature), PRCP (precipitation)
   - **VALUE**: the measurement itself (units depend on the element, often tenths)
   - **M_FLAG**: measurement flag, notes on how the value was measured (e.g. T = trace amount)
   - **Q_FLAG**: quality flag, blank if the reading passed NOAA's quality checks, a letter if it failed one
   - **S_FLAG**: source flag, which data source the reading came from
   - **OBS_TIME**: time of day the reading was taken (HHMM), often blank

- **Station metadata, ghcnd-stations.txt:**
   - **ID**: station ID (matches STATION in the yearly files)
   - **LATITUDE / LONGITUDE**: station location in decimal degrees
   - **ELEVATION**: height above sea level (m)
   - **STATE**: US state or Canadian province code (blank for other countries)
   - **NAME**: station name
   - **GSN FLAG / HCN/CRN FLAG**: whether the station belongs to certain climate monitoring networks
   - **WMO ID**: World Meteorological Organization station number

**References:**
- Full column descriptions: https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme-by_year.txt
- Element codes and units: https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme.txt (section III, "Format of data files")
- Station metadata (name, location, elevation): https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt
- Station inventory (which elements each station recorded, and for which years): https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-inventory.txt
- Inventory format: https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme.txt (section VII, "Format of ghcnd-inventory.txt")

### Initial Quality Audit

Each yearly file is very large, so we audit **one year (2020)** as a representative sample of the raw data. All yearly files share the same format.

In [8]:
import pandas as pd


url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/2020.csv.gz"
cols = ['STATION', 'DATE', 'ELEMENT', 'VALUE', 'M_FLAG', 'Q_FLAG', 'S_FLAG', 'OBS_TIME']

# Note this (1 years worth of data) takes like a minute to load
weather_2020 = pd.read_csv(url, header=None, names=cols)


url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt"
colwidths = [(0,11), (12,20), (21,30), (31,37), (38,40), (41,71), (72,75), (76,79), (80,85)]
cols = ['ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'STATE', 'NAME', 'GSN FLAG', 'HCN/CRN FLAG', 'WMO ID']

stations = pd.read_fwf(url, colspecs=colwidths, header=None, names=cols)


display(weather_2020.head())
display(stations.head())

,STATION,DATE,ELEMENT,VALUE,M_FLAG,Q_FLAG,S_FLAG,OBS_TIME
0,ASN00074087,20200101,PRCP,0,NaN,NaN,a,NaN
1,ASN00074088,20200101,PRCP,0,NaN,NaN,a,NaN
2,ASN00074100,20200101,PRCP,0,NaN,NaN,a,NaN
3,ASN00074104,20200101,PRCP,0,NaN,NaN,a,NaN
4,ASN00074106,20200101,TMAX,320,NaN,NaN,a,NaN


,ID,LATITUDE,LONGITUDE,ELEVATION,STATE,NAME,GSN FLAG,HCN/CRN FLAG,WMO ID
0,ACW00011604,17.1167,-61.7833,10.1,NaN,ST JOHNS COOLIDGE FLD,NaN,NaN,NaN
1,ACW00011647,17.1333,-61.7833,19.2,NaN,ST JOHNS,NaN,NaN,NaN
2,AE000041196,25.3330,55.5170,34.0,NaN,SHARJAH INTER. AIRP,GSN,NaN,41196.0
3,AEM00041194,25.2550,55.3640,10.4,NaN,DUBAI INTL,NaN,NaN,41194.0
4,AEM00041217,24.4330,54.6510,26.8,NaN,ABU DHABI INTL,NaN,NaN,41217.0


In [9]:
weather_2020.info()
print('\n', '-'*20, '\n')
print('Missing values per column:\n', weather_2020.isna().sum())
print('\n', '-'*20, '\n')
print('Duplicate rows:', weather_2020.duplicated().sum())
print('\n', '-'*20, '\n')
print('Number of elements:', weather_2020.ELEMENT.nunique())
print('\n', '-'*20, '\n')
print('Most common elements:\n', weather_2020.ELEMENT.value_counts().head(10))
print('\n', '-'*20, '\n')

<class 'pandas.DataFrame'>
RangeIndex: 37168585 entries, 0 to 37168584
Data columns (total 8 columns):
 #   Column    Dtype  
---  ------    -----  
 0   STATION   str    
 1   DATE      int64  
 2   ELEMENT   str    
 3   VALUE     int64  
 4   M_FLAG    str    
 5   Q_FLAG    str    
 6   S_FLAG    str    
 7   OBS_TIME  float64
dtypes: float64(1), int64(2), str(5)
memory usage: 2.2 GB

 -------------------- 

Missing values per column:
 STATION            0
DATE               0
ELEMENT            0
VALUE              0
M_FLAG      34252886
Q_FLAG      37130570
S_FLAG             0
OBS_TIME    20082888
dtype: int64

 -------------------- 

Duplicate rows: 0

 -------------------- 

Number of elements: 74

 -------------------- 

Most common elements:
 ELEMENT
PRCP    11049314
TMIN     4590596
TMAX     4567051
SNOW     4421843
SNWD     3284919
TAVG     2258576
TOBS     1716388
WESD      533002
AWND      422255
WSF2      396771
Name: count, dtype: int64

 -------------------- 



**Audit findings (2020):**
- **Size:** 37,168,585 rows × 8 columns, no duplicates
- **Missing values:** none in the core columns (STATION, DATE, ELEMENT, VALUE). M_FLAG, S_FLAG and OBS_TIME are extra notes we don't need, so their missing values don't matter
- **Long format:** each row is one element and value pair, and a row only exists if that reading was recorded. So a missing measurement shows up as a missing row, not a NaN, and "no missing values" doesn't mean every station reported every day
- **Q_FLAG:** the one flag that matters to us. 38,015 readings (about 0.1%) failed one of NOAA's quality checks and will be removed
- **Other issues:** VALUE is often in tenths (e.g. 105 = 10.5°C), and DATE is a number (YYYYMMDD) rather than a real date

## Dataset Selection: Elements, Stations and Years

The full dataset is far too large to analyze, so we narrow it down:

**Elements of interest:**
- **TMAX / TMIN / TAVG**: maximum, minimum and average daily temperature
- **PRCP**: daily precipitation
- **SNOW**: daily snowfall
- **AWND**: average daily wind speed

**Stations of interest:** one station in each of three cities with very different climates:
- **Dublin, Ireland**: mild and rainy all year
- **Toronto, Canada**: cold, snowy winters and warm summers
- **San Jose, California**: dry, sunny and mild

For each city, we pick the station that reported our elements most consistently, then use the station inventory to check how far back each station's records go. This sets our time range (see below).

In [10]:
elements = ['TMAX', 'TMIN', 'TAVG', 'PRCP', 'SNOW', 'AWND']

In [11]:
sj_stations = stations[(stations.NAME.str.split().str[:2].str.join(" ") == 'SAN JOSE') & (stations.STATE == 'CA')]
tto_stations = stations[(stations.NAME.str.split().str[0] == 'TORONTO') & (stations.STATE == 'ON')]
dub_stations = stations[(stations.NAME.str.split().str[0] == 'DUBLIN') & (stations.STATE.isna())]

display(sj_stations.head(8))
print()
display(tto_stations.head(8))
print()
display(dub_stations.head(8))   # only 1 station in Dublin

,ID,LATITUDE,LONGITUDE,ELEVATION,STATE,NAME,GSN FLAG,HCN/CRN FLAG,WMO ID
58418,US1CASC0002,37.2381,-121.8356,48.2,CA,SAN JOSE 4.6 S,NaN,NaN,NaN
58420,US1CASC0004,37.3050,-121.7750,68.0,CA,SAN JOSE 5.4 SW,NaN,NaN,NaN
58423,US1CASC0007,37.3543,-121.7955,96.9,CA,SAN JOSE 4.6 NE,NaN,NaN,NaN
58425,US1CASC0010,37.2714,-121.8765,49.4,CA,SAN JOSE 2.7 SSW,NaN,NaN,NaN
58427,US1CASC0012,37.3190,-121.7887,61.0,CA,SAN JOSE 3.5 ENE,NaN,NaN,NaN
58428,US1CASC0013,37.2733,-121.8664,49.1,CA,SAN JOSE 2.3 SSW,NaN,NaN,NaN
58432,US1CASC0018,37.3086,-121.9272,48.5,CA,SAN JOSE 3.0 WSW,NaN,NaN,NaN
58434,US1CASC0027,37.3859,-121.8240,89.3,CA,SAN JOSE 5.8 NNE,NaN,NaN,NaN


,ID,LATITUDE,LONGITUDE,ELEVATION,STATE,NAME,GSN FLAG,HCN/CRN FLAG,WMO ID
24332,CAN06158350,43.6667,-79.4000,112.5,ON,TORONTO,NaN,NaN,71266.0
24333,CAN06158355,43.6667,-79.4000,112.5,ON,TORONTO CITY,NaN,NaN,71508.0
24335,CAN0615HMAK,43.8622,-79.3700,198.1,ON,TORONTO BUTTONVILLE A,NaN,NaN,71639.0
25193,CA1ON000019,43.7144,-79.3567,124.7,ON,TORONTO 7.0 NNE - CANWARN,NaN,NaN,NaN
25477,CA1ON000564,43.6605,-79.4090,119.8,ON,TORONTO 2.2 WNW,NaN,NaN,NaN
25602,CA1ON000837,43.6894,-79.2730,137.8,ON,TORONTO 9.7 ENE,NaN,NaN,NaN
25645,CA1ON000913,43.7072,-79.4030,176.8,ON,TORONTO 6.0 NNW,NaN,NaN,NaN
31732,CAN06158359,43.6275,-79.3961,76.8,ON,TORONTO CITY CENTRE,NaN,NaN,71265.0


,ID,LATITUDE,LONGITUDE,ELEVATION,STATE,NAME,GSN FLAG,HCN/CRN FLAG,WMO ID
34121,EI000003969,53.3639,-6.3192,49.0,NaN,DUBLIN PHOENIX PARK,NaN,NaN,3969.0


In [12]:
# DUBLIN - only one station, so nothing to narrow down

dubID = dub_stations.ID.item()
dub_2020 = weather_2020[(weather_2020.STATION == dubID) & (weather_2020.ELEMENT.isin(elements))]
display(dub_2020.ELEMENT.value_counts())  
dub_2020.head()

ELEMENT
TMAX    366
TMIN    366
PRCP    366
TAVG    366
Name: count, dtype: int64

,STATION,DATE,ELEMENT,VALUE,M_FLAG,Q_FLAG,S_FLAG,OBS_TIME
45937,EI000003969,20200101,TMAX,105,NaN,NaN,E,NaN
45938,EI000003969,20200101,TMIN,8,NaN,NaN,E,NaN
45939,EI000003969,20200101,PRCP,0,NaN,NaN,E,NaN
45940,EI000003969,20200101,TAVG,44,H,NaN,S,NaN
147603,EI000003969,20200102,TMAX,117,NaN,NaN,E,NaN


In [13]:
# TORONTO

sub = weather_2020[weather_2020.STATION.isin(tto_stations.ID) & weather_2020.ELEMENT.isin(elements)]   # weather df with elements of interest and toronto stations only
counts = sub.STATION.value_counts()

ttoID = counts.idxmax()
num_reports = counts.max()
print(f'Toronto station with the most reportings in 2020: {ttoID}, with {num_reports} reportings')

tto_2020 = weather_2020[(weather_2020.STATION == ttoID) & (weather_2020.ELEMENT.isin(elements))]
display(tto_2020.ELEMENT.value_counts())
display(tto_2020.head())


# SAN JOSE

sub = weather_2020[weather_2020.STATION.isin(sj_stations.ID) & weather_2020.ELEMENT.isin(elements)]
counts = sub.STATION.value_counts()

sjID = counts.idxmax()
num_reports = counts.max()
print(f'San Jose station with the most reportings in 2020: {sjID}, with {num_reports} reportings')

sj_2020 = weather_2020[(weather_2020.STATION == sjID) & (weather_2020.ELEMENT.isin(elements))]
display(sj_2020.ELEMENT.value_counts())
display(sj_2020.head())

Toronto station with the most reportings in 2020: CAN06158731, with 1823 reportings


ELEMENT
TMAX    366
TMIN    366
PRCP    366
TAVG    366
SNOW    359
Name: count, dtype: int64

,STATION,DATE,ELEMENT,VALUE,M_FLAG,Q_FLAG,S_FLAG,OBS_TIME
20044,CAN06158731,20200101,TMAX,-2,NaN,NaN,C,NaN
20045,CAN06158731,20200101,TMIN,-32,NaN,NaN,C,NaN
20046,CAN06158731,20200101,PRCP,0,T,NaN,C,NaN
20047,CAN06158731,20200101,SNOW,0,T,NaN,C,NaN
20049,CAN06158731,20200101,TAVG,-17,NaN,NaN,C,NaN


San Jose station with the most reportings in 2020: USW00023293, with 1463 reportings


ELEMENT
TMAX    366
PRCP    366
AWND    366
TMIN    365
Name: count, dtype: int64

,STATION,DATE,ELEMENT,VALUE,M_FLAG,Q_FLAG,S_FLAG,OBS_TIME
5025,USW00023293,20200101,TMAX,156,NaN,NaN,W,2400.0
5026,USW00023293,20200101,TMIN,56,NaN,NaN,W,2400.0
5027,USW00023293,20200101,PRCP,0,NaN,NaN,W,2400.0
5032,USW00023293,20200101,AWND,29,NaN,NaN,W,NaN
106040,USW00023293,20200102,TMAX,161,NaN,NaN,W,2400.0


**Station coverage (2020 sample):**

The only Dublin station in the dataset (Dublin Phoenix Park) reports every day of the year. For Toronto and San Jose, we chose the station with the most reports on our elements.

All three stations report **TMAX, TMIN and PRCP** on nearly all 366 days of the (leap) year. Beyond that:
- **Dublin** also reports TAVG
- **Toronto** also reports TAVG and SNOW
- **San Jose** also reports AWND

This difference in reported elements gives us two routes for analysis:
- **Within each station:** look at each station over the years, including its extra elements (like snow in Toronto or wind in San Jose)
- **Across stations:** compare patterns and differences between the three cities using their shared elements (TMAX, TMIN, PRCP)

**Choosing the years:**

Now that we have our stations and elements, we need to choose our years. Our original plan was three 2-year intervals, but 2 years is probably too short to see a station's normal weather patterns, since one unusual year can skew the results. So we looked for two 5-year intervals instead.

The problem is that we don't know which years each station reported our elements consistently. That's where the station inventory file comes in: it lists, for every station and element, the first and last year the station reported it.

- Inventory format: https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme.txt (section VII, "Format of ghcnd-inventory.txt")

In [14]:
url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-inventory.txt"
colwidths = [(0,11), (12,20), (21,30), (31,35), (36,40), (41,45)]
cols = ['ID', 'LATITUDE', 'LONGITUDE', 'ELEMENT', 'FIRSTYEAR', 'LASTYEAR']

inventory = pd.read_fwf(url, colspecs=colwidths, header=None, names=cols)
inventory[inventory.ID.isin([sjID, dubID, ttoID]) & (inventory.ELEMENT.isin(elements))]

,ID,LATITUDE,LONGITUDE,ELEMENT,FIRSTYEAR,LASTYEAR
119858,CAN06158731,43.6767,-79.6306,TMAX,2013,2026
119859,CAN06158731,43.6767,-79.6306,TMIN,2013,2026
119860,CAN06158731,43.6767,-79.6306,PRCP,2013,2026
119861,CAN06158731,43.6767,-79.6306,SNOW,2013,2026
119863,CAN06158731,43.6767,-79.6306,TAVG,2013,2026
134338,EI000003969,53.3639,-6.3192,TMAX,1867,2026
134339,EI000003969,53.3639,-6.3192,TMIN,1867,2026
134340,EI000003969,53.3639,-6.3192,PRCP,1867,2026
134342,EI000003969,53.3639,-6.3192,TAVG,1973,2025
768769,USW00023293,37.3594,-121.9244,TMAX,1998,2026


**Result:** The Toronto station only started reporting in 2013, and San Jose in 1998, so two widely spaced 5-year intervals aren't possible for all three stations. Instead, we use the full **2014–2025** range. Since Toronto only began reporting in 2013, that year may not be complete, so starting from 2014 keeps things simple. All three stations report TMAX, TMIN and PRCP for every one of these years, which gives us 12 continuous years to study both seasonal patterns and year-to-year trends.